In [1]:
import sys
if sys.version_info < (3, 7):
    import contextlib
    class nullcontext:
        def __enter__(self): return self
        def __exit__(self, *args): pass
    contextlib.nullcontext = nullcontext

In [2]:
def extract_qnetwork_from_bqn(bqn_state_dict):
    qnetwork_state_dict = {}
    
    # Filter keys that start with 'q.' and remove the prefix
    for key, value in bqn_state_dict.items():
        if key.startswith('q.'):
            qnetwork_state_dict[key[2:]] = value
    
    return qnetwork_state_dict

In [3]:
from utils import QNetwork
import gym
import torch
env = gym.make("BipedalWalker-v3")
state_space = env.observation_space.shape[0]
action_space = env.action_space.shape[0]
net = QNetwork(state_space,action_space,action_scale = 6)
bqn_state_dict = torch.load('agent_1000')
qnetwork_state_dict = extract_qnetwork_from_bqn(bqn_state_dict)
net.load_state_dict(qnetwork_state_dict)

/export/kbodla/venv_3.6/lib64/python3.6/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/export/kbodla/venv_3.6/lib64/python3.6/site-packages/torch/cuda/__init__.py:143: UserWarning: 
NVIDIA RTX A6000 with CUDA capability sm_86 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_70.
If you want to use the NVIDIA RTX A6000 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(incompatible_device_warn.format(device_name, capability, " ".join(arch_list), device_name))


<All keys matched successfully>

In [4]:

import gym
import numpy as np
from tqdm import tqdm
from TD3 import TD3
import torch
#from TD3 import TD3
from PIL import Image
import os
import os
# os.environ["SDL_VIDEODRIVER"] = "dummy"
# if not os.path.exists('data/'):
#     os.mkdir('data/')

n_episodes = 100
env_name = "BipedalWalker-v3"
random_seed = 0
lr = 0.002
max_timesteps = 2000
action_scale = 6
real_action = np.linspace(-1.,1., action_scale)
# policy.actor.to('cpu')
X_train = list()
A_train = list()
#obs_train = list()
states = list()
total_reward = 0

for ep in tqdm(range(n_episodes)):
    ep_reward = 0
    state = env.reset()
    for t in range(max_timesteps):
        #obs_train.append(state)
        
        # img_array = env.render(mode='rgb_array')
        # states.append(img_array)
  
        action_prob,x = net(torch.tensor(state))
        action =  [int(x.max(1)[1]) for x in action_prob]
        new_a = np.array([real_action[x] for x in action])
        state, reward, done, _ = env.step(new_a)
        #shape_x = len(x)#.size()
        x = x.detach().cpu().numpy()
        X_train.append(x)
        A_train.append(new_a)
        ep_reward += reward        
        if done:
            break
        
    print('Episode: {}\tReward: {}'.format(ep, int(ep_reward)))
    #print("shape_state in X_train: ", shape_x)
    total_reward += ep_reward
    ep_reward = 0
env.close()        
               

print("Average Reward:", total_reward / n_episodes) 

X_train = np.array(X_train)
A_train = np.array(A_train)
#obs_train = np.array(obs_train)

obs_train = np.array(states)

# np.save('data/X_train.npy', X_train)
# np.save('data/a_train.npy', A_train)
# np.save('data/obs_train.npy', obs_train)


  1%|          | 1/100 [00:00<01:28,  1.12it/s]

Episode: 0	Reward: 302


  2%|▏         | 2/100 [00:01<01:15,  1.30it/s]

Episode: 1	Reward: 303


  3%|▎         | 3/100 [00:02<01:08,  1.41it/s]

...

Episode: 97	Reward: 304


 99%|█████████▉| 99/100 [01:05<00:00,  1.56it/s]

Episode: 98	Reward: 305


100%|██████████| 100/100 [01:05<00:00,  1.52it/s]

Episode: 99	Reward: 305
Average Reward: 302.340103982511


In [ ]:

# import gym
# import numpy as np
# from tqdm import tqdm
# from TD3 import TD3
# import torch
# #from TD3 import TD3
# from PIL import Image
# import os
# import os
# # os.environ["SDL_VIDEODRIVER"] = "dummy"
# # if not os.path.exists('data/'):
# #     os.mkdir('data/')

# n_episodes = 100

# random_seed = 0
# lr = 0.002
# max_timesteps = 2000

# # policy.actor.to('cpu')
# X_train = list()
# A_train = list()
# #obs_train = list()
# states = list()
# total_reward = 0

# for ep in tqdm(range(n_episodes)):
#     ep_reward = 0
#     state = env.reset()
#     for t in range(max_timesteps):
#         #obs_train.append(state)
        
#         # img_array = env.render(mode='rgb_array')
#         # states.append(img_array)
  
#         A, x = policy.select_action(state)
#         state, reward, done, _ = env.step(A)
#         #shape_x = len(x)#.size()
#         X_train.append(x)
#         A_train.append(A)
#         ep_reward += reward        
#         if done:
#             break
        
#     print('Episode: {}\tReward: {}'.format(ep, int(ep_reward)))
#     #print("shape_state in X_train: ", shape_x)
#     total_reward += ep_reward
#     ep_reward = 0
# env.close()        
               

# print("Average Reward:", total_reward / n_episodes) 

# X_train = np.array(X_train)
# A_train = np.array(A_train)
# #obs_train = np.array(obs_train)

# obs_train = np.array(states)

# # np.save('data/X_train.npy', X_train)
# # np.save('data/a_train.npy', A_train)
# # np.save('data/obs_train.npy', obs_train)


In [12]:
X_train[0].shape

(240,)

In [5]:
if not os.path.exists('/export/kbodla/bipedal_walker'):
    os.mkdir('/export/kbodla/bipedal_walker')

new_arr = np.expand_dims(X_train, axis=1) 
print(new_arr.shape)
np.save('/export/kbodla/bipedal_walker/X_train_new.npy', new_arr)
np.save('/export/kbodla/bipedal_walker/a_train_new.npy', A_train)

(92863, 1, 240)


In [11]:
state = env.reset()
action_prob,x = net(torch.tensor(state))
action =  [int(x.max(1)[1]) for x in action_prob]
new_a = np.array([real_action[x] for x in action])
new_a

array([ 1. , -0.2, -0.2, -1. ])

In [6]:
X_train = np.load('/export/kbodla/bipedal_walker/X_train.npy')
a_train = np.load('/export/kbodla/bipedal_walker/a_train.npy')

a_train[0]

FileNotFoundError: [Errno 2] No such file or directory: '/export/kbodla/bipedal_walker/X_train.npy'

In [17]:
A_train[0]

array([ 1. , -0.2, -0.2, -1. ])